### 1. Independent Loading and Initial Cleaning Pipelines (Reddit & Text Datasets)

In [2]:
import pandas as pd
from google.colab import drive
import re
import os

drive.mount('/content/drive')

# Define file paths
reddit_file_path = '/content/drive/MyDrive/255 Project Data/df_reddit_cleaned.csv'
text_file_path = '/content/drive/MyDrive/255 Project Data/df_text_cleaned.csv'

# --- Reddit Dataset Pipeline ---
df_reddit = pd.read_csv(reddit_file_path)
# Rename 'target' to 'label'
df_reddit = df_reddit.rename(columns={'target': 'label'})
# Convert integer labels to string for consistency with text labels
df_reddit['label'] = df_reddit['label'].astype(str)

# Map Reddit numerical labels to descriptive names
reddit_label_map = {
    '0': 'Stress',
    '1': 'Depression',
    '2': 'Bipolar',
    '3': 'Personality disorder',
    '4': 'Anxiety'
}
df_reddit['label'] = df_reddit['label'].replace(reddit_label_map)

print(f"df_reddit shape: {df_reddit.shape}")
print(f"Reddit labels: {df_reddit['label'].unique()}")

# Cleaning function for Reddit
def clean_reddit_text(text):
    text = str(text).lower() # Convert to string and lowercase
    return text

df_reddit['cleaned_text'] = df_reddit['text'].apply(clean_reddit_text)


# --- Text Dataset Pipeline ---
df_text = pd.read_csv(text_file_path)
# Rename 'status' to 'label'
df_text = df_text.rename(columns={'status': 'label'})
print(f"df_text shape: {df_text.shape}")
print(f"Text labels: {df_text['label'].unique()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
df_reddit shape: (5607, 5)
Reddit labels: ['Depression' 'Stress' 'Bipolar' 'Personality disorder' 'Anxiety']
df_text shape: (52681, 3)
Text labels: ['Anxiety' 'Normal' 'Depression' 'Suicidal' 'Stress' 'Bipolar'
 'Personality disorder']


### 2. Strip Subreddit-style Leakage Tokens (Reddit Dataset)

In [3]:
# Function to strip subreddit tokens like 'r/depression' or 'u/username'
def strip_subreddit_tokens(text):
    # Remove subreddit mentions (r/subreddit_name)
    text = re.sub(r'r/\S+', '', text, flags=re.IGNORECASE)
    # Remove user mentions (u/username)
    text = re.sub(r'u/\S+', '', text, flags=re.IGNORECASE)
    return text.strip()

print("\n--- Stripping Subreddit Tokens from Reddit Dataset ---")
df_reddit['cleaned_text'] = df_reddit['cleaned_text'].apply(strip_subreddit_tokens)

print("df_reddit after stripping:")
print(df_reddit['cleaned_text'].head())


--- Stripping Subreddit Tokens from Reddit Dataset ---
df_reddit after stripping:
0    welcome to / check-in post - a place to take a...
1    we understand that most people who reply immed...
2    anyone else just miss physical touch? i crave ...
3    i’m just so ashamed. everyone and everything f...
4    i really need a friend. i don't even have a si...
Name: cleaned_text, dtype: object


### 3. Stratified 80/20 Split on Reddit Dataset

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

print("\n--- Performing Stratified 80/20 Split on Combined Reddit and Text Matched Datasets ---")

# Select relevant columns for combination from both datasets
df_reddit_selected = df_reddit[['cleaned_text', 'label']]

# Cleaning 'statement' and creating 'cleaned_text'
df_text_processed = df_text.copy()
df_text_processed['cleaned_text'] = df_text_processed['statement'].apply(clean_reddit_text)

# Now select the desired columns from the processed df_text
df_text_selected = df_text_processed[['cleaned_text', 'label']]

# Combine the datasets
combined_data = pd.concat([df_reddit_selected, df_text_selected], ignore_index=True)
print(f"Combined data shape (Reddit + Text): {combined_data.shape}")

# Perform stratified split on the combined data
df_combined_train, df_combined_test = train_test_split(
    combined_data,
    test_size=0.2,
    random_state=42,
    stratify=combined_data['label'] # Stratify using the label column from the combined data
)

# Resetting index might be good practice after splitting
df_combined_train = df_combined_train.reset_index(drop=True)
df_combined_test = df_combined_test.reset_index(drop=True)

print(f"Combined train set shape: {df_combined_train.shape}")
print(f"Combined test set shape: {df_combined_test.shape}")

# Check stratification
print("\nCombined training set label distribution:")
print(df_combined_train['label'].value_counts(normalize=True))
print("\nCombined test set label distribution:")
print(df_combined_test['label'].value_counts(normalize=True))

# Save to parquet files
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_path = os.path.join(output_dir, 'combined_train.parquet')
test_path = os.path.join(output_dir, 'combined_test.parquet')

df_combined_train.to_parquet(train_path, index=False)
df_combined_test.to_parquet(test_path, index=False)
print(f"\nCombined training data saved to: {train_path}")
print(f"Combined testing data saved to: {test_path}")



--- Performing Stratified 80/20 Split on Combined Reddit and Text Matched Datasets ---
Combined data shape (Reddit + Text): (58288, 2)
Combined train set shape: (46630, 2)
Combined test set shape: (11658, 2)

Combined training set label distribution:
label
Depression              0.284902
Normal                  0.280377
Suicidal                0.182736
Anxiety                 0.085524
Bipolar                 0.066266
Stress                  0.063243
Personality disorder    0.036950
Name: proportion, dtype: float64

Combined test set label distribution:
label
Depression              0.284869
Normal                  0.280408
Suicidal                0.182793
Anxiety                 0.085521
Bipolar                 0.066221
Stress                  0.063218
Personality disorder    0.036970
Name: proportion, dtype: float64

Combined training data saved to: /content/drive/MyDrive/255 Project Data/combined_train.parquet
Combined testing data saved to: /content/drive/MyDrive/255 Project Data/

In [13]:
print("\n--- Creating Undersampled Training Set ---")

# Get the class distribution of the combined training data
class_counts = df_combined_train['label'].value_counts()
print("Original class distribution in df_combined_train:")
print(class_counts)

# Find the minimum class count
min_class_count = class_counts.min()
print(f"\nSmallest class count: {min_class_count}")

# Create an empty DataFrame to store the undersampled data
df_undersampled_train = pd.DataFrame()

# Undersample each class to the minimum class count
for label in class_counts.index:
    class_df = df_combined_train[df_combined_train['label'] == label]
    # Randomly sample 'min_class_count' entries from the current class
    undersampled_class = class_df.sample(n=min_class_count, random_state=42)
    df_undersampled_train = pd.concat([df_undersampled_train, undersampled_class])

# Shuffle the resulting DataFrame to mix the classes
df_undersampled_train = df_undersampled_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nShape of the undersampled training set: {df_undersampled_train.shape}")
print("\nClass distribution in df_undersampled_train (after undersampling):")
print(df_undersampled_train['label'].value_counts())

# Display the head of the new DataFrame
print("\nHead of df_undersampled_train:")
display(df_undersampled_train.head())

# Save the undersampled DataFrame to parquet
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_undersampled_path = os.path.join(output_dir, 'train_undersampled.parquet')
df_undersampled_train.to_parquet(train_undersampled_path, index=False)
print(f"\nUndersampled training data saved to: {train_undersampled_path}")


--- Creating Undersampled Training Set ---
Original class distribution in df_combined_train:
label
Depression              13285
Normal                  13074
Suicidal                 8521
Anxiety                  3988
Bipolar                  3090
Stress                   2949
Personality disorder     1723
Name: count, dtype: int64

Smallest class count: 1723

Shape of the undersampled training set: (12061, 2)

Class distribution in df_undersampled_train (after undersampling):
label
Suicidal                1723
Stress                  1723
Normal                  1723
Depression              1723
Bipolar                 1723
Anxiety                 1723
Personality disorder    1723
Name: count, dtype: int64

Head of df_undersampled_train:


,cleaned_text,label
0,i am extremely lonely. sexually frustrated. i ...,Suicidal
1,i have had the worst anxiety of my life recent...,Stress
2,"i have been trying to, but every time i fail i...",Suicidal
3,it turns out that everyone is quietly asking t...,Normal
4,i’m freaking out and i don’t know what to do. ...,Stress



Undersampled training data saved to: /content/drive/MyDrive/255 Project Data/train_undersampled.parquet


### 4. Apply SMOTE to the Training Split and Save

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
import pandas as pd
import os
import numpy as np
from IPython.display import display

print("\n--- Applying SMOTE to the Training Split (Reddit Dataset) ---")
X_train_text = df_combined_train['cleaned_text']
y_train = df_combined_train['label']

# Vectorize the text data using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_vectorized = tfidf_vectorizer.fit_transform(X_train_text)
print(f"Original training data shape (vectorized): {X_train_vectorized.shape}")

# Apply SMOTE to the vectorized data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_vectorized, y_train)
print(f"Resampled training data shape (vectorized): {X_resampled.shape}")

# Convert the resampled data back to a dense array and then to a DataFrame
X_resampled_dense = X_resampled.toarray()
feature_names = [f'tfidf_feature_{i}' for i in range(X_resampled_dense.shape[1])]
df_resampled = pd.DataFrame(X_resampled_dense, columns=feature_names)
df_resampled['label'] = y_resampled # Add the resampled labels

print(f"\nFinal df_resampled shape: {df_resampled.shape}")

# Save the resampled DataFrame to parquet
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_smote_path = os.path.join(output_dir, 'train_smote.parquet') # Changed filename to reflect vectorized data
df_resampled.to_parquet(train_smote_path, index=False)
print(f"\nResampled (vectorized) training data saved to: {train_smote_path}")
print(f"Shape of saved parquet: {df_resampled.shape}")
print("\nLabel distribution:")
print(df_resampled['label'].value_counts(normalize=True))

# --- Verification step (loading only a few columns to avoid memory issues) ---
print("\n--- Verifying Parquet file integrity (loading a few columns) ---")
try:
    # Load 'label' and a sample TF-IDF feature column
    columns_to_load = ['label', 'tfidf_feature_0']
    df_reloaded_sample = pd.read_parquet(train_smote_path, columns=columns_to_load)
    print(f"Successfully reloaded selected columns of '{train_smote_path}' with shape: {df_reloaded_sample.shape}")
    print("Sample of reloaded data (first 5 rows): ")
    display(df_reloaded_sample.head())
    print("File appears to be valid and readable.")
except Exception as e:
    print(f"Error reloading '{train_smote_path}': {e}")
    print("This indicates a problem with the file writing process or corruption within the Colab environment.")


--- Applying SMOTE to the Training Split (Reddit Dataset) ---
Original training data shape (vectorized): (46630, 5000)
Resampled training data shape (vectorized): (92995, 5000)

Final df_resampled shape: (92995, 5001)

Resampled (vectorized) training data saved to: /content/drive/MyDrive/255 Project Data/train_smote.parquet
Shape of saved parquet: (92995, 5001)

Label distribution:
label
Normal                  0.142857
Stress                  0.142857
Suicidal                0.142857
Anxiety                 0.142857
Depression              0.142857
Bipolar                 0.142857
Personality disorder    0.142857
Name: proportion, dtype: float64

--- Verifying Parquet file integrity (loading a few columns) ---
Successfully reloaded selected columns of '/content/drive/MyDrive/255 Project Data/train_smote.parquet' with shape: (92995, 2)
Sample of reloaded data (first 5 rows): 


,label,tfidf_feature_0
0,Normal,0.0
1,Stress,0.0
2,Suicidal,0.0
3,Normal,0.0
4,Anxiety,0.0


File appears to be valid and readable.


### 5. Preprocessing Report: Row Counts at Various Steps (Reddit & Text Datasets)

In [7]:
print("\n--- Preprocessing Report ---\n")

# Reddit Dataset
print("Reddit Dataset")
print(f"- Initial rows after loading: {len(df_reddit)}")
print("\nText Dataset")
print(f"- Initial rows after loading: {len(df_text)}")
print(f"- Rows in matched subset (df_text_selected): {len(df_text_selected)}")
print("\nCombined Text Dataset")
print(f"- Rows in combined dataset (df_reddit_selected + df_text_selected): {len(combined_data)}")
print(f"- Rows in combined training set (before SMOTE): {len(df_combined_train)}")
print(f"- Rows in combined test set: {len(df_combined_test)}")
print(f"- Rows in combined training set (after SMOTE): {len(df_resampled)}")



--- Preprocessing Report ---

Reddit Dataset
- Initial rows after loading: 5607

Text Dataset
- Initial rows after loading: 52681
- Rows in matched subset (df_text_selected): 52681

Combined Text Dataset
- Rows in combined dataset (df_reddit_selected + df_text_selected): 58288
- Rows in combined training set (before SMOTE): 46630
- Rows in combined test set: 11658
- Rows in combined training set (after SMOTE): 92995
